In [21]:
import os
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from scipy.signal import welch
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from torcheeg.models import DGCNN

In [22]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FS             = 128     # DEAP sampling rate (Hz)
WINDOW_SIZE    = 256     # samples per window (2 s) -> becomes DGCNN's in_channels
WINDOW_STRIDE  = 256     # non-overlapping windows

BATCH_SIZE     = 64
EPOCHS         = 30
LR             = 1e-3
WEIGHT_DECAY   = 1e-4
HID_CHANNELS   = 32
NUM_LAYERS     = 2
PATIENCE = 5
RANDOM_STATE   = 42

# Subject-wise split ratios (fraction of the 32 DEAP subjects).
# No subject appears in more than one split -- this prevents
# subject-specific information leakage between train/val/test.
N_SUBJECTS       = 32
N_TRIALS_PER_SUB = 40
TEST_SUBJ_FRAC   = 0.2   # 20% of subjects held out for testing
VAL_SUBJ_FRAC    = 0.2   # 20% of the remaining (non-test) subjects held out for validation

OUTPUT_DIR     = "gnn_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EMOTIONS       = {"Valence": 0, "Arousal": 1, "Dominance": 2, "Liking": 3}
CHANNEL_COUNTS = [32, 16, 8, 4]

# Only used to REPLICATE the Random-Forest channel ranking (for selecting
# the same channel subsets) -- it is never fed to the DGCNN itself.
BANDS = {
    "Delta": (0.5, 4), "Theta": (4, 8), "Alpha": (8, 13),
    "Beta": (13, 30), "Gamma": (30, 45),
}

print(f"Using device: {DEVICE}")


Using device: cuda


In [23]:
def load_data():
    eeg = np.load("cleaned_eeg.npy")
    labels = np.load("cleaned_labels.npy")

    n_subjects = eeg.shape[0]
    n_trials_per_subject = eeg.shape[1]

    # Reshape EEG:
    # (32 subjects, 40 trials, 32 channels, 8064 timepoints)
    # -> (1280 trials, 32 channels, 8064 timepoints)
    eeg = eeg.reshape(-1, eeg.shape[2], eeg.shape[3])

    # Reshape labels:
    # (32 subjects, 40 trials, 4 emotions)
    # -> (1280 trials, 4 emotions)
    labels = labels.reshape(-1, 4)

    # Track which subject each trial belongs to, so we can later build a
    # subject-wise train/val/test split (no subject in more than one split).
    subject_ids = np.repeat(np.arange(n_subjects), n_trials_per_subject)

    print("EEG shape         :", eeg.shape)
    print("Labels shape      :", labels.shape)
    print("Subject IDs shape :", subject_ids.shape)

    return eeg, labels, subject_ids


In [24]:
def subject_wise_split(
    n_subjects=N_SUBJECTS,
    test_frac=TEST_SUBJ_FRAC,
    val_frac=VAL_SUBJ_FRAC,
    random_state=RANDOM_STATE
):
    """
    Splits SUBJECTS (not trials) into train/val/test sets so that no
    subject's data appears in more than one split. Computed once and
    reused for every emotion and every channel count, so all
    experiments are evaluated on identical, completely unseen test
    subjects.
    """
    subject_ids = np.arange(n_subjects)

    train_val_subj, test_subj = train_test_split(
        subject_ids, test_size=test_frac, random_state=random_state
    )
    train_subj, val_subj = train_test_split(
        train_val_subj, test_size=val_frac, random_state=random_state
    )

    print(f"Train subjects ({len(train_subj)}):", sorted(train_subj.tolist()))
    print(f"Val subjects   ({len(val_subj)}):", sorted(val_subj.tolist()))
    print(f"Test subjects  ({len(test_subj)}):", sorted(test_subj.tolist()))

    return train_subj, val_subj, test_subj


In [25]:
def band_power(psd, freqs, fmin, fmax):
    idx = np.where((freqs >= fmin) & (freqs <= fmax))[0]
    return np.mean(psd[idx]) if idx.size > 0 else 0.0

In [26]:
def extract_band_power_features(eeg_data, fs=FS, bands=BANDS):
    n_samples, n_channels, _ = eeg_data.shape
    n_bands = len(bands)
    feats = np.zeros((n_samples, n_channels * n_bands))
    for i, sample in enumerate(eeg_data):
        vec = []
        for ch in range(n_channels):
            freqs, psd = welch(sample[ch], fs=fs, nperseg=fs * 2)
            for fmin, fmax in bands.values():
                vec.append(band_power(psd, freqs, fmin, fmax))
        feats[i] = vec
    return feats

In [27]:
def rank_channels_by_rf_importance(eeg_data, y, bands=BANDS):
    """
    Trains a quick Random Forest on band-power features purely to rank
    channels by importance -- this mirrors the ranking used in the RF
    experiments so the DGCNN channel-reduction study uses identical
    channel subsets.

    IMPORTANT: `eeg_data`/`y` passed in here must already be restricted to
    TRAINING-SUBJECT trials only. Ranking on the full dataset (including
    validation/test subjects) would leak test information into channel
    selection. The resulting ranking is RF-derived, not DGCNN-derived --
    report results as "DGCNN performance under RF-derived channel
    selection", not as DGCNN-generated channel importance.
    """
    X_bp = extract_band_power_features(eeg_data)
    rf = RandomForestClassifier(
        n_estimators=200, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    )
    rf.fit(X_bp, y)

    n_bands = len(bands)
    n_channels = eeg_data.shape[1]
    scores = {}
    for ch in range(n_channels):
        start = ch * n_bands
        scores[ch] = rf.feature_importances_[start:start + n_bands].sum()

    ranked = sorted(scores, key=scores.get, reverse=True)  # channel idx, best first
    return ranked

In [28]:
def get_channel_subsets(ranked_channels, counts=CHANNEL_COUNTS):
    return {n: ranked_channels[:n] for n in counts}

In [29]:
class EEGWindowDataset(Dataset):
    """
    Converts raw EEG trials of shape (n_channels, n_timepoints) into
    fixed-length windows of shape (n_channels, WINDOW_SIZE), which is
    the input format DGCNN expects: (num_electrodes, in_channels).

    norm_stats: pass the TRAIN split's (mean, std) when building the
    val/test datasets so normalisation never leaks test information.
    """

    def __init__(
        self,
        eeg_trials,
        trial_labels,
        window_size=WINDOW_SIZE,
        stride=WINDOW_STRIDE,
        norm_stats=None
    ):
        windows = []
        win_labels = []
        trial_ids = []

        for trial_id, (trial, label) in enumerate(
            zip(eeg_trials, trial_labels)
        ):
            n_timepoints = trial.shape[1]
            start = 0

            while start + window_size <= n_timepoints:
                windows.append(
                    trial[:, start:start + window_size]
                )
                win_labels.append(label)

                # Remember which original trial this window came from
                trial_ids.append(trial_id)

                start += stride

        self.windows = np.stack(windows).astype(np.float32)
        self.labels = np.array(win_labels, dtype=np.int64)
        self.trial_ids = np.array(trial_ids, dtype=np.int64)

        # Calculate normalization statistics using TRAINING data only
        if norm_stats is None:
            mean = self.windows.mean(
                axis=(0, 2),
                keepdims=True
            )

            std = (
                self.windows.std(
                    axis=(0, 2),
                    keepdims=True
                ) + 1e-8
            )

            self.norm_stats = (mean, std)

        else:
            self.norm_stats = norm_stats

        mean, std = self.norm_stats

        self.windows = (
            self.windows - mean
        ) / std

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return torch.from_numpy(
            self.windows[idx]
        ), self.labels[idx]

In [30]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += xb.size(0)

    return total_loss / total, correct / total

In [31]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_labels, all_preds, all_probs = [], [], []

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * xb.size(0)

        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = logits.argmax(dim=1)

        correct += (preds == yb).sum().item()
        total += xb.size(0)

        all_labels.extend(yb.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    avg_loss = total_loss / total
    avg_acc = correct / total

    # Calculate Macro F1-score
    avg_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return (
        avg_loss,
        avg_acc,
        avg_f1,
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs)
    )

In [32]:
@torch.no_grad()
def evaluate_trial_level(model, dataset):
    """
    Evaluate DGCNN at the original trial level.

    Each trial is divided into multiple windows.
    Predictions from all windows belonging to the same
    trial are averaged to obtain one final trial prediction.
    """

    model.eval()

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    all_probs = []
    all_labels = []

    for xb, yb in loader:

        xb = xb.to(DEVICE)

        logits = model(xb)

        # Probability of High class
        probs = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_labels.extend(
            yb.numpy()
        )

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    # Trial ID for every window
    all_trial_ids = dataset.trial_ids

    # Get unique trials
    unique_trials = np.unique(
        all_trial_ids
    )

    trial_probs = []
    trial_labels = []

    # Combine all windows belonging to each trial
    for trial_id in unique_trials:

        mask = (
            all_trial_ids == trial_id
        )

        # Average probability across windows
        avg_prob = np.mean(
            all_probs[mask]
        )

        # All windows from a trial have the same label
        trial_label = all_labels[mask][0]

        trial_probs.append(
            avg_prob
        )

        trial_labels.append(
            trial_label
        )

    trial_probs = np.array(
        trial_probs
    )

    trial_labels = np.array(
        trial_labels
    )

    # Convert probability to Low/High
    trial_preds = (
        trial_probs >= 0.5
    ).astype(int)

    return (
        trial_labels,
        trial_preds,
        trial_probs
    )

In [33]:
def run_dgcnn_experiment(
    emotion_name,
    channel_count,
    channel_idx,
    eeg_all,
    y_all,
    train_mask,
    val_mask,
    test_mask
):
    tag = f"{emotion_name.lower()}_{channel_count}ch"

    print(f"\n{'='*70}")
    print(f"  {emotion_name} | {channel_count} channels")
    print(f"{'='*70}")

    # ============================================================
    # SELECT CHANNELS
    # ============================================================

    eeg_sub = eeg_all[:, channel_idx, :]
    # Shape: (n_trials, channel_count, n_timepoints)

    # ============================================================
    # SUBJECT-WISE TRAIN / VALIDATION / TEST SPLITS
    # (train_mask/val_mask/test_mask come from subject_wise_split() in
    # main(), computed once and reused across every emotion and channel
    # count -- no subject appears in more than one split.)
    # ============================================================

    train_eeg, train_y = eeg_sub[train_mask], y_all[train_mask]
    val_eeg, val_y     = eeg_sub[val_mask],   y_all[val_mask]
    test_eeg, test_y   = eeg_sub[test_mask],  y_all[test_mask]

    print(
        f"Trials -> train: {len(train_y)} | "
        f"val: {len(val_y)} | test: {len(test_y)}"
    )

    # ============================================================
    # CREATE WINDOW DATASETS
    # ============================================================

    train_ds = EEGWindowDataset(
        train_eeg,
        train_y
    )

    val_ds = EEGWindowDataset(
        val_eeg,
        val_y,
        norm_stats=train_ds.norm_stats
    )

    test_ds = EEGWindowDataset(
        test_eeg,
        test_y,
        norm_stats=train_ds.norm_stats
    )

    # ============================================================
    # DATA LOADERS
    # ============================================================

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    # ============================================================
    # CREATE DGCNN MODEL
    # ============================================================

    model = DGCNN(
        in_channels=WINDOW_SIZE,
        num_electrodes=channel_count,
        hid_channels=HID_CHANNELS,
        num_layers=NUM_LAYERS,
        num_classes=2,
    ).to(DEVICE)

    # ============================================================
    # OPTIMIZER
    # ============================================================

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    # ============================================================
    # CLASS WEIGHTS
    # TRAINING DATA ONLY
    # ============================================================

    class_counts = np.bincount(train_y)

    class_weights = (
        len(train_y) /
        (2.0 * class_counts)
    )

    class_weights = torch.tensor(
        class_weights,
        dtype=torch.float32,
        device=DEVICE
    )

    print(
        "Training class counts:",
        class_counts
    )

    print(
        "Class weights:",
        class_weights.cpu().numpy()
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    # ============================================================
    # TRAINING HISTORY
    # ============================================================

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    # ============================================================
    # TRAINING LOOP
    # ============================================================

    for epoch in range(
        1,
        EPOCHS + 1
    ):

        tr_loss, tr_acc = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion
        )

        va_loss, va_acc, *_ = evaluate(
            model,
            val_loader,
            criterion
        )

        history["train_loss"].append(
            tr_loss
        )

        history["val_loss"].append(
            va_loss
        )

        history["train_acc"].append(
            tr_acc
        )

        history["val_acc"].append(
            va_acc
        )

        # Save best validation model
        if va_loss < best_val_loss:
            best_val_loss = va_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

        print(
            f"Epoch {epoch:3d}/{EPOCHS} | "
            f"Train Loss {tr_loss:.4f} "
            f"Acc {tr_acc:.4f} | "
            f"Val Loss {va_loss:.4f} "
            f"Acc {va_acc:.4f}"
        )

    # ============================================================
    # FINAL WINDOW-LEVEL TEST EVALUATION
    # ============================================================

    model.load_state_dict(
        best_state
    )

    _, _, _, y_true, y_pred, y_prob = evaluate(
        model,
        test_loader,
        criterion
    )

    # ============================================================
    # TRIAL-LEVEL EVALUATION
    # ============================================================

    trial_true, trial_pred, trial_prob = evaluate_trial_level(
        model,
        test_ds
    )

    trial_acc = accuracy_score(
        trial_true,
        trial_pred
    )

    trial_bal_acc = balanced_accuracy_score(
        trial_true,
        trial_pred
    )

    trial_prec = precision_score(
        trial_true,
        trial_pred,
        zero_division=0
    )

    trial_rec = recall_score(
        trial_true,
        trial_pred,
        zero_division=0
    )

    trial_f1 = f1_score(
        trial_true,
        trial_pred,
        zero_division=0
    )

    try:

        trial_auc = roc_auc_score(
            trial_true,
            trial_prob
        )

    except ValueError:

        trial_auc = float("nan")

    trial_cm = confusion_matrix(
        trial_true,
        trial_pred
    )

    # ============================================================
    # PRINT TRIAL-LEVEL RESULTS
    # ============================================================

    print("\n" + "-" * 70)

    print(
        f"TRIAL-LEVEL RESULTS | "
        f"{emotion_name} | "
        f"{channel_count} channels"
    )

    print("-" * 70)

    print(
        f"Accuracy : {trial_acc:.4f}"
    )

    print(
        f"Balanced Accuracy: "
        f"{trial_bal_acc:.4f}"
    )

    print(
        f"Precision: {trial_prec:.4f}"
    )

    print(
        f"Recall   : {trial_rec:.4f}"
    )

    print(
        f"F1-Score : {trial_f1:.4f}"
    )

    print(
        f"ROC-AUC  : {trial_auc:.4f}"
    )

    print(
        "Trial-Level Confusion Matrix:"
    )

    print(
        trial_cm
    )

    print(
        f"Number of test trials: "
        f"{len(trial_true)}"
    )

    # ============================================================
    # WINDOW-LEVEL METRICS
    # ============================================================

    acc = accuracy_score(
        y_true,
        y_pred
    )

    prec = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    rec = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    try:

        auc = roc_auc_score(
            y_true,
            y_prob
        )

    except ValueError:

        auc = float("nan")

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    print(
        f"\nTest Results | "
        f"{emotion_name} | "
        f"{channel_count} channels"
    )

    print(
        f"Accuracy : {acc:.4f}  "
        f"Precision : {prec:.4f}  "
        f"Recall : {rec:.4f}  "
        f"F1 : {f1:.4f}  "
        f"ROC-AUC : {auc:.4f}"
    )

    print(
        "Confusion Matrix:\n",
        cm
    )

    # ============================================================
    # LOSS CURVE
    # ============================================================

    epochs_range = range(1, len(history["train_loss"]) + 1)

    plt.figure(
        figsize=(7, 4)
    )

    plt.plot(
        epochs_range,
        history["train_loss"],
        label="Train Loss"
    )

    plt.plot(
        epochs_range,
        history["val_loss"],
        label="Val Loss"
    )

    plt.xlabel(
        "Epoch"
    )

    plt.ylabel(
        "Loss"
    )

    plt.title(
        f"Loss vs Epochs | "
        f"{emotion_name} | "
        f"{channel_count} ch"
    )

    plt.legend()

    plt.tight_layout()

    plt.savefig(
        f"{OUTPUT_DIR}/loss_curve_{tag}.png",
        dpi=150
    )

    plt.close()

    # ============================================================
    # ACCURACY CURVE
    # ============================================================

    plt.figure(
        figsize=(7, 4)
    )

    plt.plot(
        epochs_range,
        history["train_acc"],
        label="Train Accuracy"
    )

    plt.plot(
        epochs_range,
        history["val_acc"],
        label="Val Accuracy"
    )

    plt.xlabel(
        "Epoch"
    )

    plt.ylabel(
        "Accuracy"
    )

    plt.title(
        f"Accuracy vs Epochs | "
        f"{emotion_name} | "
        f"{channel_count} ch"
    )

    plt.legend()

    plt.tight_layout()

    plt.savefig(
        f"{OUTPUT_DIR}/accuracy_curve_{tag}.png",
        dpi=150
    )

    plt.close()

    # ============================================================
    # TRIAL-LEVEL CONFUSION MATRIX
    # ============================================================

    plt.figure(
        figsize=(5, 4)
    )

    sns.heatmap(
        trial_cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Low", "High"],
        yticklabels=["Low", "High"]
    )

    plt.xlabel(
        "Predicted"
    )

    plt.ylabel(
        "True"
    )

    plt.title(
        f"Trial-Level Confusion Matrix | "
        f"{emotion_name} | "
        f"{channel_count} ch"
    )

    plt.tight_layout()

    plt.savefig(
        f"{OUTPUT_DIR}/trial_confusion_matrix_{tag}.png",
        dpi=150
    )

    plt.close()

    # ============================================================
    # SAVE MODEL
    # ============================================================

    model_path = (
        f"{OUTPUT_DIR}/"
        f"{tag}_dgcnn_model.pth"
    )

    torch.save(
        {
            "model_state_dict": best_state,

            "channel_idx": channel_idx,

            "norm_stats": train_ds.norm_stats,

            "config": {
                "in_channels": WINDOW_SIZE,
                "num_electrodes": channel_count,
                "hid_channels": HID_CHANNELS,
                "num_layers": NUM_LAYERS,
                "num_classes": 2,
            },
        },
        model_path
    )

    print(
        f"Saved → {model_path}"
    )

    # ============================================================
    # RETURN RESULTS
    # ============================================================

    return {

        "Emotion": emotion_name,

        "Channels": channel_count,

        # Primary trial-level metrics
        "Accuracy": round(
            trial_acc,
            4
        ),

        "Balanced Accuracy": round(
            trial_bal_acc,
            4
        ),

        "Precision": round(
            trial_prec,
            4
        ),

        "Recall": round(
            trial_rec,
            4
        ),

        "F1-Score": round(
            trial_f1,
            4
        ),

        "ROC-AUC": (
            round(
                trial_auc,
                4
            )
            if not np.isnan(trial_auc)
            else trial_auc
        ),

        "Confusion Matrix": (
            trial_cm.tolist()
        ),

        # Window-level metric
        "Window Accuracy": round(
            acc,
            4
        ),

        "model_path": model_path,
    }

In [34]:
def main():
    eeg, labels, subject_ids = load_data()

    # ================================================================
    # SUBJECT-WISE SPLIT
    # Same train/validation/test subjects are reused for every
    # emotion and every channel configuration.
    #
    # This ensures:
    #   - No subject overlap between train/val/test
    #   - Direct comparison between channel configurations
    #   - Direct comparison between emotions
    # ================================================================

    train_subj, val_subj, test_subj = subject_wise_split()

    train_mask = np.isin(subject_ids, train_subj)
    val_mask   = np.isin(subject_ids, val_subj)
    test_mask  = np.isin(subject_ids, test_subj)

    all_results = []

    # ================================================================
    # RUN ALL EMOTIONS
    #
    # For each emotion:
    #   1. Create binary labels
    #   2. Perform RF channel ranking using ONLY training subjects
    #   3. Create 32/16/8/4 channel subsets
    #   4. Train and evaluate DGCNN for each channel count
    # ================================================================

    for emotion_name, col_idx in EMOTIONS.items():

        print(f"\n\n{'#'*70}")
        print(f"#  EMOTION: {emotion_name.upper()}")
        print(f"{'#'*70}")

        # ============================================================
        # CREATE BINARY EMOTION LABELS
        # Low  = score <= 5
        # High = score > 5
        # ============================================================

        y_emotion = (labels[:, col_idx] > 5).astype(int)

        print(
            "Class distribution:",
            np.bincount(y_emotion),
            "-> [Low, High]"
        )

        # ============================================================
        # RF CHANNEL RANKING
        #
        # IMPORTANT:
        # Channel ranking is calculated ONLY using training subjects.
        # Validation and test subjects are NOT used for channel
        # selection, preventing data leakage.
        #
        # Ranking is recalculated separately for each emotion.
        # ============================================================

        print(
            f"\nCalculating RF channel ranking for {emotion_name}..."
        )

        ranked_channels = rank_channels_by_rf_importance(
            eeg[train_mask],
            y_emotion[train_mask]
        )

        channel_subsets = get_channel_subsets(ranked_channels)

        print(
            f"Channel ranking complete for {emotion_name}."
        )

        # ============================================================
        # RUN 32 / 16 / 8 / 4 CHANNEL EXPERIMENTS
        # ============================================================

        for ch_count in [32, 16, 8, 4]:

            print(f"\n{'='*70}")
            print(
                f"Running {emotion_name} | "
                f"{ch_count} channels"
            )
            print(f"{'='*70}")

            ch_idx = channel_subsets[ch_count]

            result = run_dgcnn_experiment(
                emotion_name,
                ch_count,
                ch_idx,
                eeg,
                y_emotion,
                train_mask,
                val_mask,
                test_mask
            )

            all_results.append(result)

    # ================================================================
    # SUMMARY TABLE — ALL 16 EXPERIMENTS
    # ================================================================

    summary_df = pd.DataFrame([
        {
            k: v
            for k, v in r.items()
            if k not in ("Confusion Matrix", "model_path")
        }
        for r in all_results
    ])

    print("\n" + "=" * 110)
    print("DGCNN — ALL EMOTIONS | ALL CHANNEL CONFIGURATIONS")
    print("=" * 110)

    print(
        summary_df.to_string(index=False)
    )

    # ================================================================
    # SAVE COMPLETE SUMMARY
    # ================================================================

    summary_path = (
        f"{OUTPUT_DIR}/"
        f"dgcnn_all_emotions_all_channels_summary.csv"
    )

    summary_df.to_csv(
        summary_path,
        index=False
    )

    print(
        f"\nSaved → {summary_path}"
    )

    # ================================================================
    # ACCURACY VS CHANNELS
    # ONE PLOT FOR EACH EMOTION
    # ================================================================

    for emotion_name in EMOTIONS.keys():

        sub = (
            summary_df[
                summary_df["Emotion"] == emotion_name
            ]
            .sort_values("Channels")
        )

        plt.figure(figsize=(8, 5))

        plt.plot(
            sub["Channels"],
            sub["Accuracy"],
            marker="o",
            linewidth=2
        )

        for x_, v in zip(
            sub["Channels"],
            sub["Accuracy"]
        ):
            plt.annotate(
                f"{v:.3f}",
                (x_, v),
                textcoords="offset points",
                xytext=(0, 8),
                ha="center"
            )

        plt.xticks([4, 8, 16, 32])

        plt.xlabel(
            "Number of Channels"
        )

        plt.ylabel(
            "Trial-Level Test Accuracy"
        )

        plt.title(
            f"DGCNN Accuracy vs Channels | "
            f"{emotion_name}"
        )

        plt.grid(True)
        plt.tight_layout()

        plot_path = (
            f"{OUTPUT_DIR}/"
            f"dgcnn_{emotion_name.lower()}_"
            f"accuracy_vs_channels.png"
        )

        plt.savefig(
            plot_path,
            dpi=150
        )

        plt.close()

        print(
            f"Saved plot → {plot_path}"
        )

    # ================================================================
    # COMBINED ACCURACY VS CHANNELS PLOT
    # ALL EMOTIONS
    # ================================================================

    plt.figure(figsize=(9, 6))

    for emotion_name in EMOTIONS.keys():

        sub = (
            summary_df[
                summary_df["Emotion"] == emotion_name
            ]
            .sort_values("Channels")
        )

        plt.plot(
            sub["Channels"],
            sub["Accuracy"],
            marker="o",
            linewidth=2,
            label=emotion_name
        )

    plt.xticks([4, 8, 16, 32])

    plt.xlabel(
        "Number of Channels"
    )

    plt.ylabel(
        "Trial-Level Test Accuracy"
    )

    plt.title(
        "DGCNN Test Accuracy vs Channels — All Emotions"
    )

    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    combined_plot_path = (
        f"{OUTPUT_DIR}/"
        f"dgcnn_all_emotions_accuracy_vs_channels.png"
    )

    plt.savefig(
        combined_plot_path,
        dpi=150
    )

    plt.close()

    print(
        f"Saved combined plot → "
        f"{combined_plot_path}"
    )

    # ================================================================
    # BEST CHANNEL CONFIGURATION FOR EACH EMOTION
    # Based primarily on Trial-Level Accuracy
    # ================================================================

    print("\n" + "=" * 110)
    print(
        "BEST DGCNN CHANNEL CONFIGURATION "
        "FOR EACH EMOTION"
    )
    print("=" * 110)

    best_results = (
        summary_df
        .sort_values(
            ["Emotion", "Accuracy"],
            ascending=[True, False]
        )
        .groupby(
            "Emotion",
            as_index=False
        )
        .first()
    )

    print(
        best_results.to_string(index=False)
    )

    best_results.to_csv(
        f"{OUTPUT_DIR}/"
        f"dgcnn_best_channel_per_emotion.csv",
        index=False
    )

    # ================================================================
    # FINAL MESSAGE
    # ================================================================

    print("\n" + "=" * 110)
    print("ALL DGCNN EXPERIMENTS COMPLETE")
    print("=" * 110)

    print(
        "Total experiments completed:",
        len(all_results)
    )

    print(
        f"Models, metrics, summaries, and plots saved in: "
        f"{OUTPUT_DIR}/"
    )


if __name__ == "__main__":
    main()

EEG shape         : (1280, 32, 8064)
Labels shape      : (1280, 4)
Subject IDs shape : (1280,)
Train subjects (20): [0, 1, 2, 3, 4, 5, 6, 7, 10, 11, 12, 13, 14, 16, 20, 21, 22, 27, 28, 31]
Val subjects   (5): [18, 19, 23, 25, 26]
Test subjects  (7): [8, 9, 15, 17, 24, 29, 30]


######################################################################
#  EMOTION: VALENCE
######################################################################
Class distribution: [572 708] -> [Low, High]

Calculating RF channel ranking for Valence...
Channel ranking complete for Valence.

Running Valence | 32 channels

  Valence | 32 channels
Trials -> train: 800 | val: 200 | test: 280
Training class counts: [360 440]
Class weights: [1.1111112  0.90909094]
Epoch   1/30 | Train Loss 0.7021 Acc 0.5329 | Val Loss 0.6938 Acc 0.5374
Epoch   2/30 | Train Loss 0.6752 Acc 0.5763 | Val Loss 0.7166 Acc 0.4918
Epoch   3/30 | Train Loss 0.6581 Acc 0.5967 | Val Loss 0.7257 Acc 0.5189
Epoch   4/30 | Train Loss 0.6321 Acc 0